![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/transformers/onnx/HuggingFace_ONNX_in_Spark_NLP_SentenceDetectorSaT.ipynb)

## Import SaT (Segment any Text) sentence detector from HuggingFace 🤗 into Spark NLP 🚀

This notebook introduces **`SentenceDetectorSaTModel`** — a new, transformer-based sentence boundary detector in Spark NLP built on the [wtpsplit / **S**egment **a**ny **T**ext](https://github.com/segment-any-text/wtpsplit) models.

Unlike the classic rule-based `SentenceDetector` (which splits on punctuation patterns) or `SentenceDetectorDLModel` (a CNN trained mostly on European-language corpora), SaT predicts a boundary probability for **every sub-word token** with an XLM-RoBERTa backbone. In practice this means it:

- **doesn't blindly split on punctuation** — it works on noisy, lower-cased, or completely unpunctuated text (chat logs, ASR transcripts, OCR), where rule-based detectors fail;
- **generalizes across 85+ languages** out of a single model;
- **adjusts the boundary to the best spot** instead of cutting at a fixed offset, with an optional **minimum / maximum sentence length** (length-constrained Viterbi search).

A few things to keep in mind before we start 😊

- `SentenceDetectorSaTModel` is available in the Spark NLP release that introduced SaT (**6.4.3+**).
- The default model is [`segment-any-text/sat-12l-sm`](https://huggingface.co/segment-any-text/sat-12l-sm): 12 Transformer layers, **85+ languages**, MIT licensed.

## 1. Export the model to ONNX


In [1]:
%pip install -q "transformers>=4.30" torch onnx onnxruntime huggingface_hub sentencepiece wtpsplit skops

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.9/152.9 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 4.2 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer
import wtpsplit.models

In [3]:
HF_MODEL_ID  = "segment-any-text/sat-12l-sm"
TOKENIZER_ID = "FacebookAI/xlm-roberta-base"
EXPORT_DIR   = "sat-12l-sm-onnx"
ASSETS_DIR   = f"{EXPORT_DIR}/assets"
Path(ASSETS_DIR).mkdir(parents=True, exist_ok=True)

In [3]:
# 1) Load the model and export it to fp32 ONNX with a FLOAT attention mask + dynamic axes
model = AutoModelForTokenClassification.from_pretrained(HF_MODEL_ID).eval()

dummy_ids  = torch.randint(0, model.config.vocab_size, (1, 16), dtype=torch.int64)
dummy_mask = torch.ones((1, 16), dtype=torch.float32)   # SaT exports attention_mask as FLOAT

torch.onnx.export(
    model,
    ({"input_ids": dummy_ids, "attention_mask": dummy_mask},),
    f"{EXPORT_DIR}/model.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={
        "input_ids":      {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "logits":         {0: "batch", 1: "sequence"},
    },
    opset_version=14,
    do_constant_folding=True,
    dynamo=False
)

# 2) Save the tokenizer; Spark NLP reads sentencepiece.bpe.model from the assets/ folder
AutoTokenizer.from_pretrained(TOKENIZER_ID).save_pretrained(ASSETS_DIR)
!wget https://huggingface.co/FacebookAI/xlm-roberta-base/resolve/main/sentencepiece.bpe.model
!cp sentencepiece.bpe.model {ASSETS_DIR}/sentencepiece.bpe.model
print("exported ->", EXPORT_DIR)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_25650/241919981.py:18: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:218: DeprecationWarning: The feature will be removed. Please remove usage of this function
  setup_onnx_logging(verbose) as log_ctx,
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.12/dist-packages/transformers/integrations/sdpa_attention.py:77: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

exported -> sat-12l-sm-onnx


Let's confirm the folder layout. Spark NLP needs `model.onnx` at the root and `assets/sentencepiece.bpe.model` (the other tokenizer files are harmless extras).

In [4]:
!ls -lR {EXPORT_DIR}

sat-12l-sm-onnx:
total 1084064
drwxr-xr-x 2 root root       4096 Jun 17 23:13 assets
-rw-r--r-- 1 root root 1110070531 Jun 17 23:12 model.onnx

sat-12l-sm-onnx/assets:
total 16704
-rw-r--r-- 1 root root      343 Jun 17 23:13 tokenizer_config.json
-rw-r--r-- 1 root root 17098086 Jun 17 23:13 tokenizer.json


## 2. Set up Spark NLP

This part is easy via our simple Colab script.

In [8]:
! wget -q http://setup.johnsnowlabs.com/colab.sh -O - | bash

Installing PySpark 3.4.4 and Spark NLP 6.4.1
setup Colab for PySpark 3.4.4 and Spark NLP 6.4.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 MB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.4 which is incompatible.


In [ ]:
import sparknlp

# let's start Spark with Spark NLP
spark = sparknlp.start()

print("Apache Spark version: {}".format(spark.version))
print("Spark NLP version:    {}".format(sparknlp.version()))

## 3. Load the model with `loadSavedModel`

`loadSavedModel` takes the export folder and the active `SparkSession`. The key SaT parameter is `threshold`: a boundary is placed once a character's predicted probability is `>= threshold`. The default `0.25` is calibrated for `sat-12l-sm`.

In [7]:
from sparknlp.annotator import *
from sparknlp.base import *

sat = SentenceDetectorSaTModel.loadSavedModel(EXPORT_DIR, spark) \
    .setInputCols(["document"]) \
    .setOutputCol("sentence") \
    .setThreshold(0.25)

Let's save it to disk so it can be moved around and reused later via `.load()`.

In [8]:
sat.write().overwrite().save("./sat_12l_sm_spark_nlp_onnx")

Let's clean up the raw export — we don't need it anymore.

In [9]:
!rm -rf {EXPORT_DIR}

Now we can reload the saved model anywhere — on another machine, cluster, or session 😊

In [10]:
sat_loaded = SentenceDetectorSaTModel.load("./sat_12l_sm_spark_nlp_onnx") \
    .setInputCols(["document"]) \
    .setOutputCol("sentence")

> **Tip:** once a SaT model is on the [Spark NLP Models Hub](https://sparknlp.org/models), you can skip the whole import and just call the one-liner:
>
> ```python
> sat = SentenceDetectorSaTModel.pretrained("sat_12l_sm", "xx") \
>     .setInputCols(["document"]).setOutputCol("sentence")
> ```

## 4. Quick start — basic sentence segmentation

The input is a `DOCUMENT` column from `DocumentAssembler`, and the output is one `DOCUMENT` annotation per detected sentence.

In [11]:
from pyspark.ml import Pipeline

document_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

pipeline = Pipeline(stages=[document_assembler, sat_loaded])

data = spark.createDataFrame(
    [["Dr. Smith flew to Washington D.C. on Monday. He met Mr. Brown at 5 p.m. to discuss the merger."]]
).toDF("text")

result = pipeline.fit(data).transform(data)
result.selectExpr("explode(sentence.result) as sentence").show(truncate=False)

+-------------------------------------------------+
|sentence                                         |
+-------------------------------------------------+
|Dr. Smith flew to Washington D.C. on Monday.     |
|He met Mr. Brown at 5 p.m. to discuss the merger.|
+-------------------------------------------------+



## 5. SaT vs. the classic detectors — on messy text

Spark NLP already ships two sentence detectors:

- **`SentenceDetector`** — rule-based, splits on punctuation and casing patterns.
- **`SentenceDetectorDLModel`** — a CNN boundary classifier; the multilingual `xx` model covers ~18 (mostly European) languages.

Both do well on clean prose, but real-world text — chat, voice transcripts, OCR — is often **lower-cased and unpunctuated**, and they then return the whole blob as one "sentence". SaT predicts boundaries from the *language itself*. Let's run all three on the same input.

In [40]:
noisy_text = (
    "so i went to the store yesterday and grabbed some milk then i realized "
    "i forgot my wallet at home what a disaster i had to drive all the way back "
    "by the time i returned the rain had started so i just stayed in and ordered pizza"
)

# Three detectors, all reading the same 'document' column, each writing its own output column
rule_detector = SentenceDetector() \
    .setInputCols(["document"]) \
    .setOutputCol("rule_sentences")

dl_detector = SentenceDetectorDLModel.pretrained("sentence_detector_dl", "xx") \
    .setInputCols(["document"]) \
    .setOutputCol("dl_sentences")

sat_detector = SentenceDetectorSaTModel.pretrained("sat_12l_sm", "xx") \
    .setInputCols(["document"]) \
    .setOutputCol("sat_sentences")

compare_pipeline = Pipeline(stages=[document_assembler, rule_detector, dl_detector, sat_detector])

data = spark.createDataFrame([[noisy_text]]).toDF("text")
result = compare_pipeline.fit(data).transform(data)

for col_name, label in [("rule_sentences", "Rule-based SentenceDetector"),
                        ("dl_sentences",   "SentenceDetectorDLModel (xx)"),
                        ("sat_sentences",  "SentenceDetectorSaTModel")]:
    sents = [r.s for r in result.selectExpr(f"explode({col_name}.result) as s").collect()]
    print(f"{label} -> {len(sents)} sentence(s):")
    for s in sents:
        print("   •", s)
    print()

sentence_detector_dl download started this may take some time.
Approximate size to download 514.9 KB
[OK!]
sat_12l_sm download started this may take some time.
Approximate size to download 991.3 MB
[OK!]
Rule-based SentenceDetector -> 1 sentence(s):
   • so i went to the store yesterday and grabbed some milk then i realized i forgot my wallet at home what a disaster i had to drive all the way back by the time i returned the rain had started so i just stayed in and ordered pizza

SentenceDetectorDLModel (xx) -> 1 sentence(s):
   • so i went to the store yesterday and grabbed some milk then i realized i forgot my wallet at home what a disaster i had to drive all the way back by the time i returned the rain had started so i just stayed in and ordered pizza

SentenceDetectorSaTModel -> 4 sentence(s):
   • so i went to the store yesterday and grabbed some milk
   • then i realized i forgot my wallet at home
   • what a disaster i had to drive all the way back
   • by the time i returned the

The rule-based detector finds **no boundaries** (no terminal punctuation) and returns one long run-on sentence; the DL model also leans on punctuation cues and recovers none. SaT reconstructs the individual sentences from the wording alone  the robustness you need for transcripts and user-generated text.

## 6. Long-form multilingual comparison

Now let's stress all three detectors on **long, real-world paragraphs**


In [41]:
from pyspark.sql.functions import size

samples = [
    ("Portuguese",
            "el mercado de abastos abria a las seis de la manana cuando el sol apenas asomaba "
            "por encima de los tejados dona carmen ya estaba alli con su puesto de verduras "
            "habia heredado el negocio de su madre que a su vez lo habia heredado de su abuela "
            "tres generaciones de mujeres que se levantaban antes del alba para vender tomates "
            "pimientos cebollas y lechugas en el mismo rincon de la plaza mayor el olor a "
            "tierra mojada se mezclaba con el de las flores frescas que el puesto de al lado "
            "vendia los martes y los viernes los primeros clientes eran siempre los mismos "
            "la senora de la farmacia que compraba apio y zanahorias porque su marido tenia "
            "el colesterol alto el cura don esteban que pedia siempre las mejores patatas "
            "para el cocido del domingo y el maestro jubilado don aurelio que se tomaba su "
            "tiempo eligiendo las naranjas una a una apretandolas suavemente con los pulgares "
            "como si midiera la vida en kilos de fruta el barrio habia cambiado mucho en los "
            "ultimos veinte anos habian abierto tres supermercados en la misma calle y los "
            "jovenes ya no bajaban al mercado preferan pedir la compra por el movil y que "
            "se la trajeran a casa pero dona carmen no se quejaba o al menos no en voz alta "
            "sabia que el tiempo cambia las cosas y que resistir es a veces mas estupido que "
            "adaptarse pero tambien sabia que hay cosas que no se pueden comprar en una "
            "pantalla la conversacion de primera hora la cara del vecino la sensacion de "
            "tocar la fruta antes de elegirla de olerla de saber si esta madura por el peso "
            "que tiene en la mano esas cosas no tienen aplicacion no tienen algoritmo no "
            "tienen precio aunque quizas algun dia alguien lo intentara poner su nieta le "
            "habia dicho que en japon ya habia maquinas que clasificaban la fruta por "
            "dulzura usando camaras y sensores dona carmen la habia mirado sin decir nada "
            "luego habia cogido una mandarina la habia pelado despacio y se la habia dado "
            "entera sin decir ninguna palabra porque a veces la mejor respuesta es esa "
            "su madre le decia siempre que las manos dicen mas que las palabras y que el "
            "silencio bien usado vale mas que mil discursos y ella habia aprendido eso "
            "desde pequena vendiendo fruta al lado de su madre cada manana antes de ir "
            "al colegio esos recuerdos no se borran aunque pasen los anos"),

    ("Italian",
            "Il professor Andreotti entro nell'aula con dieci minuti di ritardo, il che era "
            "del tutto insolito per lui. Scusatemi, disse posando la borsa sulla cattedra "
            "con un gesto piu brusco del solito. C'e stata una riunione di dipartimento. "
            "Inutile, come sempre. Nessuno rise. Gli studenti lo conoscevano abbastanza bene "
            "da capire che quella mattina non era il caso di scherzare. Andreotti era un uomo "
            "di sessantadue anni, con i capelli bianchi e gli occhi di un grigio che sembrava "
            "cambiare tonalita a seconda dell'umore. Aveva insegnato storia medievale "
            "all'Universita La Sapienza per trentasette anni e aveva pubblicato undici "
            "monografie, l'ultima delle quali aveva vinto il Premio Viareggio nella sezione "
            "saggistica. Era, in altre parole, un uomo abituato a essere ascoltato. Oggi "
            "parliamo della caduta dell'Impero Romano d'Occidente, disse aprendo il suo "
            "taccuino sgualcito. Non la versione semplificata che avete studiato al liceo. "
            "Quella versione e sbagliata, o almeno, e incompleta in modo fuorviante. "
            "Una studentessa in prima fila, Giulia Ferri, alzo la mano. Professore, "
            "intende dire che la data del 476 d.C. e convenzionale? No, rispose lui secco. "
            "Intendo dire che e irrilevante. Il 476 e l'anno in cui Odoacre depose "
            "Romolo Augustolo. Ma l'Impero non cadde quel giorno come cade un albero colpito "
            "da un'ascia. Declino. Si sgreolo. Si trasformo, lentamente, in qualcosa di "
            "diverso. Quello che chiamiamo caduta e in realta una transizione che si svolse "
            "nell'arco di tre secoli. Si avvicino alla lavagna e scrisse in lettere maiuscole: "
            "CRISI DEL TERZO SECOLO. Tutto comincio qui, disse indicando le parole con il "
            "gessetto. Tra il 235 e il 284 d.C., l'Impero ebbe cinquanta imperatori diversi. "
            "La maggior parte mori di morte violenta. In questo periodo, il territorio "
            "imperiale fu devastato da invasioni esterne, guerre civili, pestilenze e "
            "collasso economico. La moneta si svaluto catastroficamente. Il commercio a "
            "lungo raggio si ridusse. Le citta si svuotarono. Le reti di acquedotti e strade "
            "cominciarono a deteriorarsi per mancanza di manutenzione. Ma allora perche si "
            "parla ancora del 476? chiese un altro studente dal fondo. Perche gli storici "
            "hanno bisogno di date, disse Andreotti con un sorriso amaro. Le date sono "
            "comode. Semplificano. Ma la storia non e comoda, e raramente e semplice. "
            "Il vostro compito, come storici, e imparare a vivere con l'ambiguita senza "
            "esserne paralizzati. Fece una pausa, guardo fuori dalla finestra per un momento, "
            "poi torno a guardare la classe. Cominciate a leggere Ward-Perkins. Tutto."),
    ("Turkish",
          "Iklim degisikligi, gunumuzun en acil ve karmasik sorunlarindan biri olmaya devam "
            "etmektedir. Bilim insanlari, sanayi devriminden bu yana atmosferdeki karbondioksit "
            "konsantrasyonunun yaklasik yuzde elli artarak 420 ppm'i astigini belgelemektedir. "
            "Bu artis, kuresel ortalama sicakligin 1850'li yillara kiyasla yaklasik 1,2 "
            "santigrat derece yuekselmesine yol acmistir. Bilim duenyasinin neredeyse tamami, "
            "bu isinmanin buyuk olcude insan faaliyetlerinden kaynaklandigi konusunda hem "
            "fikirdir. Turkiye, iklim degisikliginin etkilerini cesitli bicimler de "
            "hissetmektedir. Akdeniz havzasinda yer alan ulke, hem kuraklik hem de siddetli "
            "yagis olaylarinin arttigi bir cografyada bulunmaktadir. 2021 yilinda Turkiye'nin "
            "guney ve bati kiyilarini vuran orman yanginlari, ulkenin 21. yuzyilda yasadigi "
            "en buyuk yangin felaketi olarak kayitlara gecti. Ayni yil Karadeniz bolgesinde "
            "yasanan sel felaketi onlarca insanin hayatini kaybetmesine yol acti. Bu olaylar, "
            "iklim kirillganliginin soyut bir kavram olmadigini, somut ve olumcul sonuclari "
            "olan bir gerceklik oldugunu bir kez daha gozler onune serdi. Turkiye, Paris "
            "Anlasmasini 2021 yilinda onayladi ve 2053 yilina kadar karbon notrluegunu "
            "hedefledigini acikladi. Ancak elestirrmenler, mevcut politikalarin bu hedeflerle "
            "ortusemedigini one surmektedir. Enerji karmasinda komuruen payi hala onemli bir "
            "yere sahipken, yenilenebilir enerji yatirimlari son yillarda hiz kazanmistir. "
            "Ruzgar ve guenes enerjisi kapasitesi 2015'ten bu yana dort kattan fazla artmistir. "
            "Sivil toplum kuruluslari da benzer kaygilari dile getirmektedir. Kuresel duzeyde "
            "ise 2023 yili, kayitlarin tutulmaya baslandigindan bu yana en sicak yil olarak "
            "tarihe gecmistir. Dunyagenelinde asiri hava olaylarinin sikligi ve siddeti artmaya "
            "devam etmektedir. Bu baglamda, teknolojik yenilik, uluslararasi isbirligi ve "
            "bireysel davranis degisikliginin birlikte hayata gecirilmesi zorunlu gorunmektedir."
            ),
    ("Spanish",
            "el mercado de abastos abria a las seis de la manana cuando el sol apenas asomaba "
            "por encima de los tejados dona carmen ya estaba alli con su puesto de verduras "
            "habia heredado el negocio de su madre que a su vez lo habia heredado de su abuela "
            "tres generaciones de mujeres que se levantaban antes del alba para vender tomates "
            "pimientos cebollas y lechugas en el mismo rincon de la plaza mayor el olor a "
            "tierra mojada se mezclaba con el de las flores frescas que el puesto de al lado "
            "vendia los martes y los viernes los primeros clientes eran siempre los mismos "
            "la senora de la farmacia que compraba apio y zanahorias porque su marido tenia "
            "el colesterol alto el cura don esteban que pedia siempre las mejores patatas "
            "para el cocido del domingo y el maestro jubilado don aurelio que se tomaba su "
            "tiempo eligiendo las naranjas una a una apretandolas suavemente con los pulgares "
            "como si midiera la vida en kilos de fruta el barrio habia cambiado mucho en los "
            "ultimos veinte anos habian abierto tres supermercados en la misma calle y los "
            "jovenes ya no bajaban al mercado preferan pedir la compra por el movil y que "
            "se la trajeran a casa pero dona carmen no se quejaba o al menos no en voz alta "
            "sabia que el tiempo cambia las cosas y que resistir es a veces mas estupido que "
            "adaptarse pero tambien sabia que hay cosas que no se pueden comprar en una "
            "pantalla la conversacion de primera hora la cara del vecino la sensacion de "
            "tocar la fruta antes de elegirla de olerla de saber si esta madura por el peso "
            "que tiene en la mano esas cosas no tienen aplicacion no tienen algoritmo no "
            "tienen precio aunque quizas algun dia alguien lo intentara poner su nieta le "
            "habia dicho que en japon ya habia maquinas que clasificaban la fruta por "
            "dulzura usando camaras y sensores dona carmen la habia mirado sin decir nada "
            "luego habia cogido una mandarina la habia pelado despacio y se la habia dado "
            "entera sin decir ninguna palabra porque a veces la mejor respuesta es esa "
            "su madre le decia siempre que las manos dicen mas que las palabras y que el "
            "silencio bien usado vale mas que mil discursos y ella habia aprendido eso "
            "desde pequena vendiendo fruta al lado de su madre cada manana antes de ir "
            "al colegio esos recuerdos no se borran aunque pasen los anos"
            ),
]

lang_df1 = spark.createDataFrame(samples, ["lang", "text"])
counted1 = compare_pipeline.fit(lang_df1).transform(lang_df1)

counted1.select(
    "lang",
    size("rule_sentences.result").alias("Rule"),
    size("dl_sentences.result").alias("DL_xx"),
    size("sat_sentences.result").alias("SaT"),
).show(truncate=False)

+----------+----+-----+---+
|lang      |Rule|DL_xx|SaT|
+----------+----+-----+---+
|Portuguese|1   |1    |27 |
|Italian   |41  |38   |41 |
|Turkish   |18  |17   |17 |
|Spanish   |1   |1    |27 |
+----------+----+-----+---+



## 7. Controlling sentence length — with boundary *adjustment*, not blind cutting

Sometimes you need to cap (or floor) sentence length  e.g. to fit a downstream model's context window. The classic detector's `splitLength` **forcibly** cuts at a fixed offset regardless of meaning.

SaT instead does a **length-constrained Viterbi search**: set `minSentenceLength` and/or `maxSentenceLength` (in characters) and it finds the globally highest-probability set of boundaries such that *every* segment respects the bounds. When a length cap is active, `threshold` is ignored. The result is segments that stay within your limits **and** land on the most natural boundary available.

In [15]:
EN_TEXT = "Modern smartphones have quietly become the most important computing devices in human history. A typical flagship released in 2024 ships with a 6.7 inch OLED display, a chipset built on a 3 nm process, and at least 12 GB of RAM. Photography is where the marketing battles are fiercest. Manufacturers now advertise 200 MP main sensors, periscope zoom lenses, and computational night modes that stack dozens of frames in under a second. Battery technology, by contrast, has improved far more slowly. Most phones still rely on lithium ion cells rated around 5,000 mAh, although charging speeds have leapt forward dramatically. Some brands now claim a full charge in roughly twenty minutes. Software support has also become a genuine differentiator. A few vendors promise up to seven years of operating system updates, which would have been unthinkable a decade ago. Repairability, however, remains a sore point for consumers and regulators alike. Glued batteries, fragile glass backs, and proprietary screws make do it yourself fixes difficult. The European Union has responded with right to repair rules and a mandate for USB C charging. Whether these regulations meaningfully change buyer behaviour is still an open question. For now, the average person upgrades roughly every three years, often nudged along by a carrier contract rather than a failing device."

sat_bounded = sat_loaded \
    .setInputCols(["document"]) \
    .setOutputCol("sentence") \
    .setMinSentenceLength(80) \
    .setMaxSentenceLength(160)

pipeline = Pipeline(stages=[document_assembler, sat_bounded])
data = spark.createDataFrame([[EN_TEXT]]).toDF("text")
result = pipeline.fit(data).transform(data)

sentences = [r.s for r in result.selectExpr("explode(sentence.result) as s").collect()]
for i, s in enumerate(sentences, 1):
    print(f"[{len(s):>3} chars] sentence {i}: {s}")

[ 93 chars] sentence 1: Modern smartphones have quietly become the most important computing devices in human history.
[133 chars] sentence 2: A typical flagship released in 2024 ships with a 6.7 inch OLED display, a chipset built on a 3 nm process, and at least 12 GB of RAM.
[128 chars] sentence 3: Photography is where the marketing battles are fiercest. Manufacturers now advertise 200 MP main sensors, periscope zoom lenses,
[139 chars] sentence 4: and computational night modes that stack dozens of frames in under a second. Battery technology, by contrast, has improved far more slowly.
[125 chars] sentence 5: Most phones still rely on lithium ion cells rated around 5,000 mAh, although charging speeds have leapt forward dramatically.
[121 chars] sentence 6: Some brands now claim a full charge in roughly twenty minutes. Software support has also become a genuine differentiator.
[116 chars] sentence 7: A few vendors promise up to seven years of operating system updates, which would have b

Every segment lands inside the `[80, 160]` character window, yet the cuts still fall on real sentence boundaries rather than mid-word — that's the boundary *adjustment* at work. Tighten or loosen `minSentenceLength` / `maxSentenceLength` to trade segment granularity against your downstream constraints.

## Parameter reference

| Parameter | Default | What it does |
| --- | --- | --- |
| `threshold` | `0.25` | Boundary probability cutoff (use `0.025` for `sat-12l`). Ignored when a length bound is set. |
| `minSentenceLength` | `0` | Minimum characters per sentence. Activates length-constrained (Viterbi) segmentation. |
| `maxSentenceLength` | `0` | Maximum characters per sentence. Activates length-constrained (Viterbi) segmentation. |
| `blockSize` | `510` | Real sub-word tokens per ONNX window (max 510 for XLM-R). |
| `stride` | `256` | Token stride between overlapping windows (smaller = more overlap, smoother boundaries). |
| `satBatchSize` | `8` | Windows per ONNX forward pass. |
| `weighting` | `"hat"` | Overlap weighting: `"hat"` (trust window centres) or `"uniform"`. |
| `trimWhitespace` | `True` | Strip leading/trailing whitespace from each sentence. |
| `explodeSentences` | `False` | Put each sentence on its own DataFrame row. |

That's it! You've exported a SaT model to ONNX, imported it into Spark NLP 🚀